In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
# 【关键两行】——必须同时设置！
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti TC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题[1][2][3][4][5][6][7][8][9][10]

import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/聚宽数据')
import datasource 
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.getAStockCodes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:4938,第一支股票:000001.XSHE


index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-06      0  6.45  6.58  6.33   6.49  8.520715e+08   
            2018-07-09      1  6.51  6.76  6.50   6.76  1.255008e+09   
            2018-07-10      2  6.76  6.76  6.66   6.73  8.030390e+08   
            2018-07-11      3  6.56  6.61  6.50   6.58  7.447658e+08   
            2018-07-12      4  6.54  6.82  6.52   6.75  1.008659e+09   

                             volume  
code        date                     
000001.XSHE 2018-07-06  131957580.0  
            2018-07-09  188260099.0  
            2018-07-10  119750900.0  
            2018-07-11  113666923.0  
            2018-07-12  149984167.0

In [3]:
# 这里用kama来
dt.loc[:, 'real_price']=(dt['open'] + dt['high'] + dt['low'] + dt['close'] )/4 # 这里是
# 然后其他均线
ma_timeperiod = [3,5,10,20,30,60]
for i in ma_timeperiod:
    dt.loc[:, f'MA{i}']=talib.MA(dt['close'], timeperiod=i)
dt.tail()

index   open   high    low  close         money  \
code        date                                                          
000001.XSHE 2026-09-23   1995  11.43  11.46  11.30  11.35  1.052015e+09   
            2026-09-24   1996  11.35  11.47  11.29  11.30  1.186737e+09   
            2026-09-28   1997  11.28  11.41  11.27  11.30  8.105758e+08   
            2026-09-29   1998  11.30  11.41  11.28  11.35  7.846328e+08   
            2026-09-30   1999  11.36  11.65  11.33  11.57  1.205815e+09   

                             volume  real_price        MA3     MA5    MA10  \
code        date                                                             
000001.XSHE 2026-09-23   92669758.0     11.3850  11.430000  11.420  11.479   
            2026-09-24  104381872.0     11.3525  11.370000  11.408  11.450   
            2026-09-28   71534143.0     11.3150  11.316667  11.378  11.431   
            2026-09-29   69097909.0     11.3350  11.316667  11.352  11.407   
            2026-09-30  104535745.0     11.4775  11.406667  11.374  11.407   

                           MA20       MA30       MA60  
code        date                                       
000001.XSHE 2026-09-23  11.4995  11.367000  11.050000  
            2026-09-24  11.4975  11.376667  11.070667  
            2026-09-28  11.4925  11.391000  11.091167  
            2026-09-29  11.4865  11.407333  11.109167  
            2026-09-30  11.4820  11.432667  11.131333

In [4]:
# 这里计算一下相关系数，首先提取矩阵
_columns = ['real_price']
_columns.extend([f'MA{i}' for i in  ma_timeperiod])
dt2 = dt.loc[:, _columns]
dt2.corr() # 相关系数

,real_price,MA3,MA5,MA10,MA20,MA30,MA60
real_price,1.000000,0.999118,0.997022,0.990746,0.979198,0.968209,0.936076
MA3,0.999118,1.000000,0.998878,0.993446,0.982099,0.971181,0.939108
MA5,0.997022,0.998878,1.000000,0.996654,0.986046,0.975379,0.943450
MA10,0.990746,0.993446,0.996654,1.000000,0.993779,0.984347,0.953533
MA20,0.979198,0.982099,0.986046,0.993779,1.000000,0.996074,0.970458
MA30,0.968209,0.971181,0.975379,0.984347,0.996074,1.000000,0.983325
MA60,0.936076,0.939108,0.943450,0.953533,0.970458,0.983325,1.000000


跟均线3最接近，想想也是，均线本身是根据收盘价得到的，而真实价格很接近收盘价。  
顺便看到均线也是数字相近的相关系数高